In [ ]:
import asyncio
import itertools
from datetime import datetime, timezone

import pandas as pd
import joblib

DATA_FOLDER = "PW_0.5h"
TEST_PATH = f"{DATA_FOLDER}/test.csv"

df = pd.read_csv(TEST_PATH)

# Your dataset's last column is the machine state indicator
LABEL_COL = df.columns[-1]

# Load a previously trained model or preprocessing pipeline
model = joblib.load("anomaly_model.joblib")

# These must match the model's training features and order
FEATURE_COLS = [
    "CMS air pressure",
    "Concentrazione ossigeno base",
    "Nitrogen pressure"
]

queue = asyncio.Queue(maxsize=100)
results = []

In [ ]:
async def producer(interval=0.2, n_events=100):
    records = df.to_dict("records")

    for i, reading in zip(
        range(n_events),
        itertools.cycle(records)
    ):
        event = {
            "id": i,
            "arrival_time": datetime.now(
                timezone.utc
            ).isoformat(),
            "features": {
                c: reading[c] for c in FEATURE_COLS
            },
            "machine_state": reading[LABEL_COL]
        }

        await queue.put(event)
        await asyncio.sleep(interval)

    await queue.put(None)

In [ ]:
async def consumer():
    while True:
        event = await queue.get()

        try:
            if event is None:
                break

            X = pd.DataFrame(
                [event["features"]],
                columns=FEATURE_COLS
            )

            prediction = model.predict(X)[0]

            results.append({
                "id": event["id"],
                "time": event["arrival_time"],
                "prediction": prediction,
                "machine_state": event["machine_state"]
            })

        finally:
            queue.task_done()

In [ ]:
await asyncio.gather(
    producer(interval=0.2, n_events=100),
    consumer()
)

results_df = pd.DataFrame(results)
display(results_df.tail(10))